<div style='font-size: 30px; color: #000000;background-color: #9BEBA5;border: 3px solid black;text-align: center;'>
<div><b>Inductive learning scheme <br> for indoor positioning systems </b></div>
</div>

### Table of contents
0. [Environment setup](#environment-setup)
1. [Data preprocessing and graph construction](#graph-construction)
2. [Optimization](#optimization)
3. [Training & Testing](#training_testing)

___

# 0. Environment setup

## Module loading

In [1]:
import os
import sys
import time
import glob
import json
import ast
import re
import importlib
from tqdm.notebook import tqdm
import matplotlib.pyplot as plt
import pandas as pd
import itertools 
from itertools import product
import optuna
import torch

sys.path.append(os.path.abspath('../src'))

# Custom libraries
import indoorloc_enums
indoorloc_enums = importlib.reload(indoorloc_enums)
import indoorloc_data
indoorloc_data = importlib.reload(indoorloc_data)
import indoorloc_models
indoorloc_models = importlib.reload(indoorloc_models)
import indoorloc_trainer
indoorloc_trainer = importlib.reload(indoorloc_trainer)
import indoorloc_optimizer
indoorloc_optimizer = importlib.reload(indoorloc_optimizer)
import indoorloc_viz
indoorloc_viz = importlib.reload(indoorloc_viz)

import indoorloc_data as ildata
from indoorloc_data import (
    IndoorLocDataset, 
    IndoorLocGraphDataLoader, 
    IndoorLocPreprocessor, 
    IndoorLocGraphData
)
import indoorloc_viz as ilviz
from indoorloc_models import (
    SAGEClassifier,
    SAGERegressor,
)
import indoorloc_trainer as iltrainer
from indoorloc_trainer import (
    GNNClassificationTrainer,
    GNNRegressionTrainer
)
import indoorloc_optimizer as iloptimizer
from indoorloc_optimizer import (
    GNNRegressionOptimizer
)

## Environment information

System characteristics and CUDA availability check.

In [94]:
ilviz.EnvironmentInfo().show()


##################################################
	ENVIRONMENT INFORMATION
##################################################

Operating System: Linux
CPU: AMD Ryzen 5 2600 Six-Core Processor
RAM: 15.56 GB
--------------------------------------------------
Selected device cuda
CUDA version: 12.6
Number of available GPUs: 1
GPU 0: NVIDIA GeForce RTX 2060
--------------------------------------------------


----

# 1. Data preprocessing and graph construction

Data preprocessing and graph construction for each dataset from the grid’s parameter combinations.

Available datasets:
* **UJIIndoorLoc**: 'UJI1',
* **UTSIndoorLoc**: 'UTS1',
* **SODIndoorLoc**: 'SOD01', 'SOD02', 'SOD06',
* **TUTDatasets**: 'SAH1', 'TIE1', 'TUT1', 'TUT2', 'TUT3', 'TUT4', 'TUT5'

Available parameters:
* `normalization`: 'lineal', 'powed'
* `pca_components`: Between 0.0 and 1.0
* `distance_metric`: 'manhattan', 'cosine'
* `n_neighbors`: At least 1 is required
* `graph_scheme`: 'transductive', 'inductive'

In [164]:
# Select dataset(s)
dataset_selection = ['TIE1']
datasets = {name: name for name in dataset_selection}

# Select learning scheme (inductive or transductive)
learning_scheme = "inductive"

# Generate parameter combinations|
parameter_grid = {
    'normalization': ['powed'],
    'pca_components': [0.9],
    'distance_metric': ['cosine'],
    'n_neighbors': [25]
}

parameter_combinations = list(product(
    *parameter_grid.values()
))

# For each dataset and parameter combination, generate graph data (with n splits)
n_splits = 10
graph_datas = {}
for dataset in tqdm(datasets, desc="Selecting dataset"):
    data_splits = []
    graph_datas[dataset] = []
    dataset_structure = 'sodindoorloc' if dataset.startswith('SOD') else 'ujiindoorloc'

    for norm, pca_components, metric, n_neighbors in tqdm(
        parameter_combinations, 
        desc=f"Creating graph data"
    ):
        # Load dataset
        df_original = IndoorLocDataset(
            dataset_structure=dataset_structure,
            path=f'../data/{dataset}/{dataset}', 
            header=None
        )

        # Preprocess dataset
        df_preprocessed = IndoorLocPreprocessor().preprocess_dataset(
            data=df_original, 
            normalization=norm,
            pca_components=pca_components
        )
        
        # Construct graph
        for i in tqdm(range(n_splits), desc=f"Splitting data"):
            # Construct graph
            graph_data = IndoorLocGraphData().create_data_loader(
                dataset=df_preprocessed, 
                val_size=0.2,
                n_split=i,
                graph_params={
                    'metric': metric, 
                    'k': n_neighbors,
                    'scheme': learning_scheme
                }
            )
            data_splits.append(graph_data)
    
        # Append graph data
        graph_datas[dataset].append({
            'parameters':{
                'pca_components': pca_components,
                'normalization': norm,
                'metric': metric,
                'n_neighbors': n_neighbors,
            },
            'splits': data_splits}
        )


Selecting dataset:   0%|          | 0/1 [00:00<?, ?it/s]

Creating graph data:   0%|          | 0/1 [00:00<?, ?it/s]

Splitting data:   0%|          | 0/10 [00:00<?, ?it/s]

**Transductive scheme**

In [6]:
gviz = indoorloc_viz.GraphVisualizer()

Example of graph data encapsulated in the `PyTorch Geometric Data` object for the classification task.

In [ ]:
graph_datas['TUT1'][0]['transductive'].cls

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4)) 

clusters = ['class', 'split']
graph_data = graph_datas['SOD01'][3]['transductive'].cls

for ax, cluster in zip(axes, clusters):
    gviz.draw_graph(graph_data, scheme="transductive", cluster=cluster, ax=ax) 
    
plt.tight_layout()
plt.show()

**Inductive scheme**

In [ ]:
gdata = graph_datas["TUT2"][0]['splits'][0].cls

fig, axes = plt.subplots(1, 3, figsize=(18, 6)) 

splits = ['train', 'val', 'test']

for ax, split in zip(axes, splits):
    gviz.draw_graph(gdata[split], scheme="inductive", cluster="class", ax=ax) 
    ax.set_title(f"{split} split", fontsize=20)

plt.tight_layout()
plt.show()

# 3. Training & Testing

### Training

In [165]:
# Select task
task = 'classification'
# Select dataset
dataset = 'TIE1'
# Select graph
n_graph = 0
# Select data splits
graph_data = graph_datas[dataset][n_graph]['splits']
# Select models parameters
model_params = {
    'input_dim': iltrainer.get_num_features(graph_data[0], learning_scheme),
    'hidden_dim': [256, 256],
    'output_dim': iltrainer.get_num_classes(graph_data[0], learning_scheme) if task == 'classification' else 1, 
    'n_layers': 2,
    'dropout': [0.7],
    'learning_rate': 1e-3,
    'optim_factor': 0.9,
    'weight_decay': 1e-4,
    'mlp_layers': 2
}

for split in tqdm(range(len(graph_data)), desc="Training model"):
    model = SAGEClassifier(**model_params) if task == 'classification' else SAGERegressor(**model_params)
    trainer = GNNClassificationTrainer().train_validate(
        data=graph_data[split].cls, 
        model=model,
        verbose=2,
        max_epochs=400,
        patience=100,
        show_train_process=False
    ) if task == 'classification' else GNNRegressionTrainer().train_validate(
        data=graph_data[split].reg, 
        model=model,
        verbose=2,
        max_epochs=800,
        patience=200,
        show_train_process=False
    )
    
    model_path = os.path.join("../models", f"{learning_scheme}_{model.__class__.__name__}_{dataset}_g{n_graph}_s{split}.pt")
    torch.save(model.state_dict(), model_path)

Training model:   0%|          | 0/10 [00:00<?, ?it/s]

### Testing (Inference)

In [166]:
predictions = []
for split in tqdm(range(len(graph_data)), desc="Testing model"):
    prediction = GNNClassificationTrainer().test(
        data=graph_data[split].cls if task == 'classification' else graph_data[split].reg, 
        model=SAGEClassifier(**model_params) if task == 'classification' else SAGERegressor(**model_params),
        pretrained_model=f"../models/{learning_scheme}_{model.__class__.__name__}_{dataset}_g{n_graph}_s{split}.pt"   
    )
    
    predictions.append(prediction)

Testing model:   0%|          | 0/10 [00:00<?, ?it/s]

#### Results

In [167]:
iltrainer.summarize_predictions(predictions=predictions, 
                               graph_params=graph_datas[dataset][n_graph]['parameters'], 
                               model_params=model_params,
                               task=task,
                               save_path=f"../results/{learning_scheme}_{model.__class__.__name__}_{dataset}.csv")

,timestamp,accuracy_mean,accuracy_std,elapsed_time_mean,elapsed_time_std
0,2026-03-03 03:16:56,0.004,0.0084,0.0093,0.0007
